# Fun

Purpose of this notebook is to actually do training to discover different model 
configurations that would work to train a pretty good TinyGPT. 

In [10]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


cuda
NVIDIA A100-SXM4-80GB


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
!git clone https://github.com/sanchitram1/242b-hw3
%cd 242b-hw3

Cloning into '242b-hw3'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 83 (delta 47), reused 61 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 542.96 KiB | 1.19 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/242b-hw3


In [3]:
from tokenizer import build_tokenizer
from training import train_model
import json
from utils import save_json, generate_text
from plot import plot_training_curves, plot_validation_curves
from tokenizer import count_tokens, build_token_memmap
from models import TokenChunkDataset, model_checkpoint_path, load_model
from config import (
    DataConfig,
    GlobalTrainingConfig,
    RunConfig,
    TokenConfig,
    TokenizationConfig,
    ModelConfig,
)
from IPython.display import Markdown, Image

In [4]:
token_config = TokenConfig()
data_config = DataConfig()
global_training_config = GlobalTrainingConfig()
tokenization_config = TokenizationConfig()
run_config = RunConfig("experiments-pt3-context=512")

Directory already exists: /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/experiments-pt3-context=512


In [6]:
tokenizer = build_tokenizer(
    tokenization_config, token_config, data_config.training_file_colab
)

Found it in /content/drive/MyDrive/courses/242B/HW3/artifacts/shared/tokenizers/tinystories_bpe_metaspace_5000_1000000.json, loading from there


In [7]:
train_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.training_file_colab
)
valid_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.validation_file_colab
)
print(f"There are {train_token_count} tokens in train, {valid_token_count} valid")

train_token_memmap_path = build_token_memmap(
    tokenization_config,
    token_config,
    tokenizer,
    data_config.training_file_colab,
    train_token_count,
)
valid_token_memmap_path = build_token_memmap(
    tokenization_config,
    token_config,
    tokenizer,
    data_config.validation_file_colab,
    valid_token_count,
)

There are 176680096 tokens in train, 4850935 valid


In [8]:
train_dataset = TokenChunkDataset(
    train_token_memmap_path, train_token_count, global_training_config.context_length
)
valid_dataset = TokenChunkDataset(
    valid_token_memmap_path, valid_token_count, global_training_config.context_length
)

In [11]:
configs = [
    ModelConfig(
        name="large-plus",
        d_model=384,
        n_heads=6,
        n_layers=6,
        d_ff=1536,
        batch_size=16,
        learning_rate=2.5e-4,
        weight_decay=0.1,
        warmup_steps=1000,
        max_steps=100000,
        use_amp=torch.cuda.is_available(),
    ),
    ModelConfig(
        name="xlarge",
        d_model=512,
        n_heads=8,
        n_layers=6,
        d_ff=2048,
        batch_size=64,
        learning_rate=2.5e-4,
        weight_decay=0.1,
        warmup_steps=1000,
        max_steps=100000,
        use_amp=torch.cuda.is_available(),
    ),
    ModelConfig(
        name="xlarge-plus",
        d_model=768,
        n_heads=12,
        n_layers=6,
        d_ff=3072,
        batch_size=64,
        learning_rate=2.5e-4,
        weight_decay=0.1,
        warmup_steps=1000,
        max_steps=100000,
        use_amp=torch.cuda.is_available(),
    ),
]


In [ ]:
results: dict[str, dict] = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for config in configs:
    metrics_path = run_config.metrics / f"{config.name}.json"
    if metrics_path.exists():
        results[config.name] = json.loads(metrics_path.read_text(encoding="utf-8"))
        continue
    result = train_model(
        run_config,
        global_training_config,
        config,
        tokenizer,
        train_dataset,
        valid_dataset,
        device=device,
    )
    results[config.name] = result
    save_json(result, metrics_path)

    if device.type == "cuda":
        torch.cuda.empty_cache()

plot_training_curves(results, run_config.plots / "training_loss_all_models.png")
plot_validation_curves(results, run_config.plots / "validation_loss_all_models.png")

[large-plus] Loss(step=1)=8.60
[large-plus] Loss(step=10000)=2.50
[large-plus] Loss(step=20000)=2.22
[large-plus] Loss(step=30000)=2.00
[large-plus] Loss(step=40000)=1.93
[large-plus] Loss(step=50000)=1.83
[large-plus] Loss(step=60000)=1.97
[large-plus] Loss(step=70000)=1.87
[large-plus] Loss(step=80000)=1.87
[large-plus] Loss(step=90000)=1.68
[large-plus] Loss(step=100000)=1.83
[xlarge] Loss(step=1)=8.59
[xlarge] Loss(step=10000)=1.92
[xlarge] Loss(step=20000)=1.69
[xlarge] Loss(step=30000)=1.70
[xlarge] Loss(step=40000)=1.65
[xlarge] Loss(step=50000)=1.54
[xlarge] Loss(step=60000)=1.54
[xlarge] Loss(step=70000)=1.52
[xlarge] Loss(step=80000)=1.50
[xlarge] Loss(step=90000)=1.47
[xlarge] Loss(step=100000)=1.49
[xlarge-plus] Loss(step=1)=8.71
[xlarge-plus] Loss(step=10000)=1.84


In [ ]:
SAMPLE_PROMPTS = (
    "Once upon a time",
    "A little girl and her pet cat",
    "A fairy woke up",
    "A lion, a bear, and a penguin",
    "The two children",
)

In [8]:
SAMPLE_PROMPTS = (
    "Early one morning",
    "One day, a group of friends went to the park.",
    "Tom and his best friend Brian were playing with their cars.",
    "A cat named Mia wanted to go outside.",
)


In [ ]:
generations: list[dict] = []
output_settings = [
    {"temperature": 0.7, "top_k": 30},  # balanced
    {"temperature": 0.5, "top_k": 10},  # conservative
    {"temperature": 0.8, "top_k": 50},  # experimental
]

for prompt in SAMPLE_PROMPTS:
    for config in configs:
        model_path = model_checkpoint_path(run_config, config)
        model = load_model(model_path, device)

        for setting in output_settings:
            temperature = setting["temperature"]
            top_k = setting["top_k"]
            generation = generate_text(
                token_config,
                global_training_config,
                model,
                tokenizer,
                prompt,
                device,
                256,
                temperature,
                top_k,
            )

            generations.append(
                {
                    "model": config.name,
                    "prompt": prompt,
                    "generated_text": generation,
                    "temperature": temperature,
                    "top_k": top_k,
                }
            )

samples_dir = run_config.run_dir / "samples"
samples_dir.mkdir(parents=True, exist_ok=True)

generations_path = samples_dir / "new-generations.json"
generations_path.write_text(
    json.dumps(generations, indent=2),
    encoding="utf-8",
)

print(f"Saved generations to {generations_path}")


Saved generations to /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/experiments-pt2-vocab=5000/samples/new-generations.json


In [ ]:
from dataclasses import asdict

manifest = {
    "run_id": run_config.run_id,
    "run_dir": str(run_config.run_dir),
    "device": str(device),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "data": {
        "training_file": str(data_config.training_file),
        "validation_file": str(data_config.validation_file),
    },
    "tokenization": asdict(tokenization_config),
    "global_training": asdict(global_training_config),
    "models": [asdict(config) for config in configs],
    "artifacts": {
        "models_dir": str(run_config.models),
        "metrics_dir": str(run_config.metrics),
        "plots_dir": str(run_config.plots),
        "samples_dir": str(samples_dir),
        "generations": str(generations_path),
    },
    "generation_settings": output_settings,
}

manifest_path = run_config.run_dir / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(f"Saved manifest to {manifest_path}")


Saved manifest to /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/experiments-pt2-vocab=5000/manifest.json


In [ ]:
for sample in generations:
    name = sample["model"]
    display(
        Markdown(
            f"**{name}**: Prompt=`{sample['prompt']}` *({sample['temperature']}, {sample['top_k']})*"
        )
    )
    display(Markdown(sample["generated_text"]))


**large-plus**: Prompt=`Early one morning` *(0.7, 30)*

Early one morning she saw a big red ball. She felt a bit scared, so she picked it up and gave it a big hug.
Unfortunately, the ball began to bounce around in two. She wanted to have it, but it was too late.
She tried to reach it but it was too high. Then, a friendly dog came by. The dog was very kind and it licked the ball with its paws. The ball kept bouncing and until it was all better. 
The ball was so thankful that it had been able to play with the ball. She was so happy that she had made it to the other side of the room. 
The moral of this story is that it is important to always be careful when trying to catch things. For a little help from then on, no matter how hard or you get to enjoy the fun and have fun.

**large-plus**: Prompt=`Early one morning` *(0.5, 10)*

Early one morning when she was out of bedroom. She was so excited!
At night, when she was sleeping, she went to sleep, dreaming of more fun days she had with her mom.

**large-plus**: Prompt=`Early one morning` *(0.8, 50)*

...

A cat named Mia wanted to go outside. She walked to the door and saw a big box. Mia was very curious, so she went inside the box. Inside, she found a lot of toys.
Mia played with the toys and had lots of fun. Then, Mia saw her mom and dad come to the door. They were happy to see Mia playing. Mia's mom and dad were surprised too. They all played together and had a great day.

**xlarge-plus**: Prompt=`A cat named Mia wanted to go outside.` *(0.5, 10)*

A cat named Mia wanted to go outside. She put on her coat because it was cold outside. Mia saw her friend, a dog named Max, and said, "Max, let's go play!"
Mia and Max played in the snow. They made a big snowman. They had fun. But then, they saw a big hill. Mia and Max wanted to go up the hill. They did not know how to go up the hill.
Mia had an idea. She told Max to rub his back on the icy hill. Max did what Mia said. He rubbed his back on the icy hill. Max was happy. They went up the hill together. They had fun.

**xlarge-plus**: Prompt=`A cat named Mia wanted to go outside.` *(0.8, 50)*

A cat named Mia wanted to go outside. She asked her friend, Tom, to come with her. "Tom, come with me to the park!" Mia said. Tom came and said, "Yes, let's go!"
As they walked, they saw a big tree. Mia had an apple in her hand. But when they got closer, the apple fell from the tree. They were surprised and laughed. They went back to the park and had a fun day together.